# Hinglish Embeddings Workshop

**Can a machine cluster everyday conversations — without being told the topic?**

In this workshop we'll:
1. Load 2,000+ Hinglish conversational exchanges
2. Convert each conversation to a **384-number vector** (an embedding)
3. Compress those vectors to 2D so we can *see* them
4. Let the machine automatically discover clusters — no topic labels used
5. Ask a question in Hindi/Hinglish and find matching conversations

> **Runtime**: ~8 minutes on Colab free tier (CPU). Go to **Runtime → Change runtime type → T4 GPU** for ~3 minutes.
>
> Run all cells in order: **Runtime → Run all**

## 1 — Install libraries

We need four packages:
- `sentence-transformers` — loads the multilingual embedding model
- `datasets` — loads the Hinglish dataset from Hugging Face in one line
- `umap-learn` — compresses 384D embeddings to 2D
- `plotly` — interactive charts (hover to see conversation details)

<details>
<summary><strong>🔵 Teaching moment: what is each library actually doing?</strong></summary>

<strong>sentence-transformers</strong> is a wrapper around Hugging Face's transformers library. It makes it easy to load models specifically designed to produce <em>sentence-level</em> embeddings — as opposed to word-level embeddings (like Word2Vec) or token-level outputs (like raw BERT). Under the hood it runs a transformer neural network and mean-pools the token outputs into one vector per sentence.

<br><br>

<strong>datasets</strong> connects to the Hugging Face Hub — a public repository of thousands of ML datasets. With one call (load_dataset(...)) it streams or downloads the data, caches it locally, and hands you a Python object that behaves like a list of dictionaries. No manual CSV download needed.

<br><br>

<strong>umap-learn</strong> implements UMAP (Uniform Manifold Approximation and Projection). It's a dimensionality reduction algorithm — similar in goal to PCA, but better at preserving the <em>local structure</em> of data (i.e. which points are neighbours of which). We use it to squash 384 dimensions down to 2 so we can draw a picture.

<br><br>

<strong>plotly</strong> produces interactive HTML charts. Unlike matplotlib, Plotly charts let users zoom, pan, and hover over individual dots to see the data behind them — essential for exploring a 2,000-point scatter plot.

</details>

In [ ]:
!pip install sentence-transformers datasets umap-learn plotly pandas fuzzywuzzy python-Levenshtein numpy scikit-learn -q

## 2 — Load and explore the Hinglish dataset

The dataset has natural Hindi-English conversational exchanges from everyday life.
We sample 2,500 conversations to keep things fast.

**Before we cluster**: look at the language mix and conversation lengths — what patterns exist?

<details>
<summary><strong>🔵 Teaching moment: Hinglish and code-switching</strong></summary>

<strong>What is Hinglish?</strong> It's a real language phenomenon where Hindi and English are mixed within the same sentence or conversation. Example: "Mujhe ek coffee dedo, please!" (Give me a coffee, please!). It's extremely common in urban India, among young people, in tech companies, and online. Code-switching (mixing languages) is a sign of multilingual fluency, not confusion.

<br><br>

<strong>Why is this dataset special?</strong> Most ML datasets are either pure English or pure Hindi. This dataset reflects <em>real conversation</em> — the way millions of Indians actually talk. Training embeddings on real code-switched text means the model learns that Hindi, English, and Hinglish variations of the same idea are semantically equivalent.

<br><br>

<strong>Language detection without labels</strong> We won't manually label conversations as "50% Hindi, 50% English". Instead, we'll let the model's clusters reveal patterns. Some clusters might be dominated by Hindi-heavy conversations; others by English-heavy; others truly mixed. This is emergent structure from data.

</details>

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

print("Loading Hinglish conversations dataset...")
ds = load_dataset("Abhishekcr448/Hinglish-Everyday-Conversations-1M", split="train")
df = ds.to_pandas().sample(2500, random_state=42).reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head(5)

In [ ]:
# Inspect the data structure and prepare for analysis
print("First 3 conversations:\n")
for i in range(3):
    print(f"[{i}]")
    print(f"  {df.iloc[i]['english']}")
    print(f"  Hindi: {df.iloc[i]['hinglish']}")
    print()

# Create a combined conversation column for embedding
df["conversation"] = df["english"].fillna("") + " " + df["hinglish"].fillna("")
df["conversation_length"] = df["conversation"].str.len()

# Estimate language mix (rough heuristic: count common Devanagari characters)
def estimate_hindi_ratio(text):
    """Estimate % of text that appears to be in Devanagari script (Hindi)."""
    if not isinstance(text, str):
        return 0.0
    # Devanagari Unicode range: U+0900 to U+097F
    hindi_chars = sum(1 for c in text if '\u0900' <= c <= '\u097F')
    total_chars = max(len(text), 1)
    return (hindi_chars / total_chars) * 100

df["hindi_ratio"] = df["conversation"].apply(estimate_hindi_ratio)

print(f"Dataset prepared. Total conversations: {len(df)}")
print(f"\nConversation length (characters):")
print(f"  Min: {df['conversation_length'].min()}")
print(f"  Max: {df['conversation_length'].max()}")
print(f"  Mean: {df['conversation_length'].mean():.0f}")
print(f"\nLanguage mix (estimated % Hindi):")
print(f"  Min: {df['hindi_ratio'].min():.1f}%")
print(f"  Max: {df['hindi_ratio'].max():.1f}%")
print(f"  Mean: {df['hindi_ratio'].mean():.1f}%")

In [ ]:
# Visualise conversation length distribution
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Conversation length histogram
axes[0].hist(df["conversation_length"], bins=50, color="#3498db", edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Conversation Length (characters)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Conversation Lengths")
axes[0].grid(alpha=0.3)

# Language mix histogram
axes[1].hist(df["hindi_ratio"], bins=50, color="#e74c3c", edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Estimated Hindi Ratio (%)")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of Language Mix (Hindi %)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Conversations by language mix category:")
mostly_english = (df["hindi_ratio"] < 20).sum()
mostly_hinglish = ((df["hindi_ratio"] >= 20) & (df["hindi_ratio"] < 60)).sum()
mostly_hindi = (df["hindi_ratio"] >= 60).sum()
print(f"  Mostly English (<20% Hindi): {mostly_english}")
print(f"  Mixed Hinglish (20-60% Hindi): {mostly_hinglish}")
print(f"  Mostly Hindi (>=60% Hindi): {mostly_hindi}")

## 3 — Embed the conversations

We use **`paraphrase-multilingual-MiniLM-L12-v2`** — a model trained on 50+ languages including Hindi.
It converts any text to a vector of **384 numbers**.
Think of each number as measuring one abstract "dimension of meaning".
Two conversations about greetings will have similar numbers; a greeting and a complaint will differ.

> **Why multilingual?** So Cell 8 works — you can type a Hindi query and it lands in the same space as the English conversations.

<details>
<summary><strong>🔵 Teaching moment: what is an embedding, really?</strong></summary>

<strong>The core idea</strong> An embedding is a function that maps <em>anything</em> (a word, a sentence, an image, a user's watch history) to a fixed-length list of numbers. The numbers aren't random — they're chosen so that <em>similar things produce similar numbers</em>. That's it. That one property is what makes embeddings powerful.

<br><br>

<strong>Where do those 384 numbers come from?</strong> The model is a transformer neural network with ~12 layers. Each layer transforms the input a little more. The final output is a 384-dimensional vector. The model was trained (by someone else, not us) on millions of sentence pairs labelled as "similar" or "different". Through training, it learned to compress meaning into 384 numbers in a way that preserves similarity. We just use it — like a calculator.

<br><br>

<strong>Why 384 dimensions?</strong> It's a design choice by the model's creators. Larger models use 768 or 1536 dimensions and capture more nuance; smaller models use fewer and run faster. 384 is a sweet spot for speed vs. quality. The exact value doesn't matter much for understanding the concept.

<br><br>

<strong>The multilingual trick</strong> paraphrase-multilingual-MiniLM-L12-v2 was trained on text in 50+ languages <em>mapped to the same embedding space</em>. This means "hello" in English and "नमस्ते" in Hindi end up as nearby vectors. So when you embed a Hindi query in Cell 8, it lands near English conversations with similar meaning — cross-language similarity for free.

<br><br>

<strong>What does "inference" mean?</strong> We are <em>not</em> training the model — we're running it on new inputs. Training would mean adjusting the model's weights, which requires days on expensive hardware. Inference (what we do here) just runs a forward pass through the fixed network — fast even on a laptop CPU.

</details>

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("Embedding 2,500 Hinglish conversations...")
embeddings = model.encode(
    df["conversation"].tolist(),
    show_progress_bar=True,
    batch_size=64
)

print(f"\nEmbedding shape: {embeddings.shape}")
print(f"→ {embeddings.shape[0]} conversations, each represented by {embeddings.shape[1]} numbers")

## 4 — Compress to 2D with UMAP

We can't plot 384 dimensions. UMAP squashes them to 2 while trying to keep "nearby" points nearby.

**Analogy**: imagine flattening a 3D globe onto a 2D map. Some distortion is unavoidable, but countries that were neighbours stay neighbours.
UMAP does the same for meaning-space.

<details>
<summary><strong>🔵 🟢 Teaching moment: UMAP vs PCA — why does it matter?</strong></summary>

<strong>Why not just use PCA?</strong> PCA (Principal Component Analysis) is the classical method for dimensionality reduction. It finds the directions of maximum <em>global</em> variance and projects data onto them. It's fast and interpretable, but it's a <em>linear</em> method — it can only find straight-line structure. If the data curves or folds in high-dimensional space, PCA will distort it badly.

<br><br>

UMAP is non-linear. It builds a graph of nearest neighbours in high-dimensional space, then finds a 2D layout that preserves those neighbourhood relationships as faithfully as possible. The result looks much more like the real structure of the data.

<br><br>

<strong>The parameters we set:</strong><br>
• n_neighbors=15 — how many nearest neighbours to consider when building the graph. Higher = more global structure preserved; lower = more fine-grained local clusters.<br>
• min_dist=0.1 — how tightly points can be packed together in 2D. Lower values create tighter, more separated clusters.<br>
• metric="cosine" — how we measure distance between embeddings. Cosine similarity measures the <em>angle</em> between vectors, which is better than Euclidean distance for high-dimensional text embeddings (where all vectors are roughly the same length).

<br><br>

<strong>Why does UMAP take longer than the embedding step?</strong> Embedding is a single forward pass through a neural network — highly parallelisable on GPU or CPU. UMAP builds a nearest-neighbour graph over all 2,500 points, which requires comparing every point to many others — it scales roughly as O(N × log N). For 2,500 points it takes ~1-2 minutes; for 50,000 points it would take ~10 minutes.

<br><br>

<strong>Important caveat</strong> The 2D positions are a <em>projection</em> — exact distances in 2D don't perfectly reflect exact distances in 384D. Don't read too much into the precise coordinates. What matters is the overall cluster structure and which conversations are in the same neighbourhood.

</details>

In [ ]:
import umap

print("Running UMAP (this takes ~1-2 minutes on CPU)...")
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)
coords = reducer.fit_transform(embeddings)

df["x"] = coords[:, 0]
df["y"] = coords[:, 1]

print(f"Done. Each conversation now has an (x, y) position in 2D space.")
df[["conversation", "conversation_length", "hindi_ratio", "x", "y"]].head()

## 5 — Auto-cluster with KMeans

KMeans divides the 2,500 conversations into 8 groups based purely on their 384D embeddings.

**We never mentioned conversation topics.** The algorithm only sees numbers — yet it groups conversations that are semantically similar. Let's see what kinds of conversations cluster together.

<details>
<summary><strong>🔵 🟢 Teaching moment: how does KMeans work, and what is unsupervised learning?</strong></summary>

<strong>KMeans in plain English</strong><br>
1. Place 8 "centre" points randomly in 384D space.<br>
2. Assign every conversation to its nearest centre.<br>
3. Move each centre to the average position of the conversations assigned to it.<br>
4. Repeat steps 2–3 until the centres stop moving.<br>
The result: 8 groups where conversations within a group are close to each other in embedding space, and groups are as far apart from each other as possible.

<br><br>

<strong>Why 8 clusters?</strong> We chose 8 as a starting point — a reasonable number for everyday conversation diversity. In real applications, you'd use methods like the <em>elbow method</em> or <em>silhouette score</em> to find the best K. Try changing 8 to 5 or 12 in Cell 5 and re-running — the clusters will reorganise.

<br><br>

<strong>Supervised vs. unsupervised learning</strong><br>
• Supervised: you give the algorithm labelled examples ("this is greetings", "this is food talk") and it learns to predict labels for new data. Requires human annotation.<br>
• Unsupervised: you give the algorithm only the data, no labels. It finds structure on its own. That's what we're doing here — KMeans has never seen topic labels.

<br><br>

The fact that unsupervised clusters often align with natural conversation categories (like greetings, complaints, food, work) is what makes embeddings so powerful: the model has learned a representation of meaning that mirrors how humans categorise things, without being explicitly taught those categories.

<br><br>

<strong>Note on n_init=10</strong> KMeans is sensitive to where the initial centres are placed. We run it 10 times with different random starts and keep the best result. This is why you'll see it's slightly slower than you might expect.

</details>

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=8, random_state=42, n_init=10)
df["cluster"] = km.fit_predict(embeddings).astype(str)

print("Cluster sizes:")
print(df["cluster"].value_counts().sort_index())

# Peek at what's in each cluster — sample one conversation per cluster
print("\nSample conversation from each cluster:")
for c in sorted(df["cluster"].unique()):
    sample = df[df["cluster"] == c].iloc[0]
    preview = sample["conversation"][:80] + "..."
    hindi_pct = f"{sample['hindi_ratio']:.0f}%"
    print(f"  Cluster {c} ({hindi_pct} Hindi): {preview}")

In [ ]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

def find_conversations(query_snippet, threshold=60, limit=10):
    """
    🟣 Fuzzy search utility: find conversations by approximate text match.
    
    Use this to search for conversations by keywords without needing the exact text.
    Handles typos, partial phrases, and variations.
    
    Args:
        query_snippet (str): Conversation text (partial or full, case-insensitive)
        threshold (int): Match confidence 0-100. Lower = more lenient. Default: 60
        limit (int): Max results to return. Default: 10
    
    Returns:
        DataFrame: Matching conversations with similarity scores, sorted best-match first
    
    Examples:
        find_conversations("hello")                    # Search for greetings
        find_conversations("coffee")                   # Search for food talk
        find_conversations("meeting", threshold=50)    # Relaxed threshold
    """
    convs = df["conversation"].tolist()
    
    # Use fuzzywuzzy to find close matches
    matches = process.extract(
        query_snippet,
        convs,
        scorer=fuzz.token_sort_ratio,
        limit=limit
    )
    
    # Filter by threshold and convert to dataframe
    results = []
    for conv, score in matches:
        if score >= threshold:
            row = df[df["conversation"] == conv].iloc[0]
            results.append({
                "match_score": score,
                "cluster": row["cluster"],
                "hindi_ratio": f"{row['hindi_ratio']:.0f}%",
                "length": row["conversation_length"],
                "conversation_preview": row["conversation"][:150] + "..."
            })
    
    if not results:
        print(f"No conversations found matching '{query_snippet}' (threshold={threshold})")
        return None
    
    return pd.DataFrame(results)

# Test the fuzzy search
print("🟣 Fuzzy Search Utility Loaded\n")
print("Try these examples:")
print("  find_conversations('hello')        # Greeting-like queries")
print("  find_conversations('coffee')       # Food/drink queries")
print("  find_conversations('meeting')      # Work-related queries")
print("\nOr search for anything: find_conversations('your_text')")

## 6 — Interactive map (coloured by cluster)

Each dot is a conversation. Hover over any dot to read the full text, language mix, and cluster assignment.

**Look for**: do clusters look visually tight? Do nearby dots look semantically similar when you read them?

In [ ]:
import plotly.express as px

# Create preview text for hover
df["conv_preview"] = df["conversation"].str[:100] + "…"
df["hindi_str"] = df["hindi_ratio"].apply(lambda x: f"{x:.0f}%")

fig = px.scatter(
    df,
    x="x", y="y",
    color="cluster",
    hover_data={
        "conv_preview": True,
        "hindi_str": True,
        "conversation_length": True,
        "x": False,
        "y": False
    },
    labels={
        "conv_preview": "Conversation",
        "hindi_str": "Language Mix",
        "conversation_length": "Length (chars)"
    },
    title="Hinglish Conversations — Embedding Space (UMAP) · Coloured by Auto-Discovered Cluster",
    width=950, height=720
)
fig.update_traces(marker=dict(size=5, opacity=0.75))
fig.update_layout(legend_title_text="Cluster", hovermode="closest")
fig.show()

## 6b — Interactive 3D visualization

Explore the embedding space in **3D** with full rotation and zoom.
Each dot is positioned in 3D space — hover to see conversation details, rotate to see different angles.

**Better for understanding cluster geometry**: the 2D projections flatten some structure. 3D lets you see which conversations are truly neighbours in high-dimensional space.

<details>
<summary><strong>🟡 Tip: how to use the 3D plot</strong></summary>

<strong>Controls:</strong><br>
• Drag to rotate<br>
• Scroll to zoom<br>
• Right-click + drag to pan<br>
• Hover over dots to see conversation text, language mix, length<br>
• Click legend items to toggle clusters on/off

<br><br>

<strong>What to look for:</strong><br>
• Are clusters visually separated in 3D space?<br>
• Do clusters that look mixed in 2D (Cell 6) have clearer separation in 3D?<br>
• Which conversations form tight, well-defined regions vs. spread-out clouds?

<br><br>

<strong>Why 3D matters:</strong> UMAP's 2D projection is a lossy compression. The 3D version preserves more of the neighbourhood structure from the original 384D space. You might see clusters that looked fuzzy in 2D become crisp in 3D.

</details>

In [ ]:
import plotly.graph_objects as go
from sklearn.decomposition import PCA

print("Computing 3D projection (PCA on embeddings)...")
pca_3d = PCA(n_components=3, random_state=42)
coords_3d = pca_3d.fit_transform(embeddings)

print(f"3D coordinates computed. Variance explained: {pca_3d.explained_variance_ratio_.sum():.1%}")

df["z"] = coords_3d[:, 2]

# Create much shorter text preview for tooltips
df["conv_tiny"] = df["conversation"].str[:60] + "…"

# Create 3D scatter plot
fig_3d = go.Figure()

# Add a trace for each cluster
for cluster in sorted(df["cluster"].unique()):
    cluster_data = df[df["cluster"] == cluster]
    
    fig_3d.add_trace(go.Scatter3d(
        x=cluster_data["x"],
        y=cluster_data["y"],
        z=cluster_data["z"],
        mode="markers",
        marker=dict(size=4, opacity=0.7),
        text=cluster_data["conv_tiny"],
        customdata=np.column_stack((
            cluster_data["hindi_str"],
            cluster_data["conversation_length"],
            cluster_data["cluster"]
        )),
        hovertemplate="<b>%{text}</b><br>Language: %{customdata[0]}<br>Length: %{customdata[1]} chars<br>Cluster: %{customdata[2]}<extra></extra>",
        name=f"Cluster {cluster}",
        showlegend=True
    ))

fig_3d.update_layout(
    title="Hinglish Conversations — 3D Embedding Space (PCA) · Rotatable",
    scene=dict(
        xaxis_title="UMAP Axis 1",
        yaxis_title="UMAP Axis 2",
        zaxis_title="PCA Axis 3",
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.2)
        )
    ),
    width=1000,
    height=800,
    hovermode="closest",
    font=dict(size=11)
)

fig_3d.show()

## 7 — Same map, coloured by conversation length

Now we colour the **exact same dots** by their conversation length (short, medium, long).

**Key question**: Do conversations of similar length cluster together, or are they spread across the map?
If spread across: length and semantic meaning are independent. If together: certain topics tend to have longer or shorter exchanges.

<details>
<summary><strong>🟢 Teaching moment: the "aha" moment — what patterns tell us</strong></summary>

<strong>What you're looking for</strong> Compare the two charts side by side (or toggle between them):
<br>
• Cell 6: colours = machine-discovered clusters (0–7)<br>
• Cell 7: colours = conversation length (short/medium/long)

<br><br>

If one colour in Cell 7 dominates a single cluster in Cell 6 — certain conversation topics tend to be brief or lengthy. For example, greetings might be short, while complaints or stories might be long.

<br><br>

<strong>Why this is interesting</strong> The clustering algorithm had no information about length. It clustered purely on semantic meaning. The fact that length patterns emerge shows that conversations are structured by both what people talk about <em>and</em> how much time they spend discussing it. Real data is multifaceted.

<br><br>

<strong>Why won't it be a perfect match?</strong><br>
1. Semantic similarity and length are partially independent — a short greeting and a long greeting are still both greetings, but in different clusters because of embedding noise.<br>
2. UMAP introduces some distortion in the 2D projection.<br>
3. KMeans assumes spherical clusters — real semantic clusters are often irregular shapes.

<br><br>

A good partial match is actually the expected outcome, and it's still a meaningful result.

</details>

In [ ]:
# Bin conversations by length
df["length_category"] = pd.cut(
    df["conversation_length"],
    bins=[0, 200, 400, float('inf')],
    labels=["Short (<200)", "Medium (200-400)", "Long (>400)"]
)

fig2 = px.scatter(
    df,
    x="x", y="y",
    color="length_category",
    hover_data={
        "conv_preview": True,
        "hindi_str": True,
        "conversation_length": True,
        "x": False,
        "y": False
    },
    labels={
        "length_category": "Conversation Length",
        "conv_preview": "Conversation",
        "hindi_str": "Language Mix"
    },
    title="Same Map — Coloured by Conversation Length",
    width=950, height=720
)
fig2.update_traces(marker=dict(size=5, opacity=0.75))
fig2.update_layout(hovermode="closest")
fig2.show()

## 8 — Try it yourself (Hindi / Hinglish / English)

Type **any description or phrase** — in Hindi, Hinglish, or English.
The model embeds your query into the same 384D space and finds the 5 nearest conversations.

**Examples to try**:
- `"नमस्ते, कैसे हो?"` — Hello, how are you?
- `"coffee ke liye kahan jaaye?"` — Where should we go for coffee?
- `"meeting mein kya hua?"` — What happened in the meeting?
- `"दुकान कहाँ है?"` — Where is the shop?
- `"I'm so tired today"` — Simple English query

<details>
<summary><strong>🔵 🟢 Teaching moment: cosine similarity and nearest-neighbour search</strong></summary>

<strong>How does "find nearest conversations" work?</strong> When you type a query, the model converts it to a 384D vector — exactly like it did for the conversations. We then measure the <em>similarity</em> between your query vector and every conversation's vector. The 5 with the highest similarity scores are returned.

<br><br>

<strong>What is cosine similarity?</strong> Cosine similarity measures the <em>angle</em> between two vectors, not their length:
<br>
cosine_similarity(A, B) = (A · B) / (|A| × |B|)
<br><br>
• Score = 1.0 → vectors point in the same direction → same meaning<br>
• Score = 0.0 → vectors are perpendicular → unrelated meaning<br>
• Score = -1.0 → vectors point opposite directions → opposite meaning
<br><br>
We use cosine (not Euclidean distance) because text embeddings tend to cluster on the surface of a high-dimensional sphere — what matters is direction, not magnitude.

<br><br>

<strong>Why does Hindi work?</strong> Because this model was trained on parallel data (the same sentences in multiple languages) so Hindi and English sentences with the same meaning produce similar vectors. Your Hindi query vector will be close to the Hinglish conversation vectors that describe the same kind of exchange. This is called a <em>multilingual embedding space</em>.

<br><br>

<strong>What would break it?</strong><br>
• A query that describes something completely outside the training data (e.g. a very specific technical term the model hasn't seen)<br>
• A very short, ambiguous query like "hi" — too many conversations start with a greeting<br>
• A query in an unusual dialect or with heavy accent spelling

<br><br>

<strong>This pattern has a name: semantic search</strong> Keyword search (like old Google) looks for exact word matches. Semantic search finds meaning-matches even with different words. Almost every modern search system — Google, YouTube, Spotify, Amazon — uses embedding-based semantic search under the hood.

</details>

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# ← Change this to any description you like!
query = "नमस्ते, कैसे हो?"

q_emb = model.encode([query])
sims = cosine_similarity(q_emb, embeddings)[0]
top_idx = sims.argsort()[-5:][::-1]

print(f"Query: {query}")
print(f"\nTop 5 nearest conversations:\n")

results = df.iloc[top_idx][["conv_preview", "hindi_str", "conversation_length"]].copy()
results.columns = ["Conversation Preview", "Language Mix", "Length (chars)"]
results["Similarity Score"] = sims[top_idx].round(3)
results = results.reset_index(drop=True)
results.index += 1
display(results)

In [ ]:
# Visualise WHERE your query lands on the map
q_2d = reducer.transform(q_emb)

import plotly.graph_objects as go

fig3 = px.scatter(
    df, x="x", y="y",
    color="cluster",
    opacity=0.4,
    hover_data={"conv_preview": True, "hindi_str": True, "conversation_length": True, "x": False, "y": False},
    title=f'Your query lands here: "{query}"',
    width=950, height=720
)
fig3.update_traces(marker=dict(size=4))

# Mark the query point
fig3.add_trace(go.Scatter(
    x=[q_2d[0, 0]], y=[q_2d[0, 1]],
    mode="markers+text",
    marker=dict(size=18, color="black", symbol="star"),
    text=["← your query"],
    textposition="middle right",
    name="Query",
    showlegend=True
))

# Mark the top 5 nearest conversations
near = df.iloc[top_idx]
fig3.add_trace(go.Scatter(
    x=near["x"], y=near["y"],
    mode="markers+text",
    marker=dict(size=12, color="red", symbol="circle-open", line=dict(width=2)),
    text=near["conv_preview"].str.slice(0, 20),
    textposition="top center",
    name="Top 5 matches",
    showlegend=True
))

fig3.update_layout(hovermode="closest")
fig3.show()

## 9 — (Optional) Save your results

Save the 2D coordinates + clusters + metadata as a Parquet file.
If you have a Hugging Face account you can upload your own version of the dataset.

In [ ]:
out = df[[
    "conversation",
    "conversation_length",
    "hindi_ratio",
    "x", "y", "z",
    "cluster",
    "length_category"
    ]].copy()
out.to_parquet("hinglish_conversations_2d.parquet", index=False)
print("Saved to hinglish_conversations_2d.parquet")
print(f"Rows: {out.shape[0]}, Columns: {out.shape[1]}")

# To upload to Hugging Face (needs HF account + token):
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_file(
#     path_or_fileobj="hinglish_conversations_2d.parquet",
#     path_in_repo="hinglish_conversations_2d.parquet",
#     repo_id="your-username/hinglish-embeddings",
#     repo_type="dataset",
# )

## 10 — Discussion: Where does this show up in real life?

| Application | What embeddings do |
|-------------|-------------------|
| **Google Assistant** | Embed your voice query; find relevant commands or responses |
| **WhatsApp Filters** | Embed message; check if it's similar to known spam |
| **Instagram Search** | Embed your text query; find posts with similar captions |
| **Spotify Recommendations** | Embed songs you like; suggest nearby songs in meaning-space |
| **Chat Bots** | Embed user messages; retrieve relevant responses from memory |
| **Content Moderation** | Embed user post; compare to known harmful content |

Every time a system needs to understand *meaning* — not just match keywords — it probably uses embeddings.

### Things to explore

- Change `n_clusters` in Cell 5 from 8 to 12 or 5. How do the clusters reorganise?
- Try a completely wrong query (e.g. "quantum physics"). Where does it land? Does it find any matches?
- Find a conversation you recognise or understand well. Does it cluster with similar-sounding conversations?
- Compare Cell 6 (cluster colours) with Cell 7 (length colours). Do certain clusters tend to be longer or shorter?
- Try a code-mixed query like "main aaj coffee leke meetings attend karunga" (I'll attend meetings with coffee today). Does it match relevant conversations?